In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 50

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers
dense1 = keras.layers.Dense(128, activation='relu')(concat)
dense2 = keras.layers.Dense(64, activation='relu')(dense1)
dense3 = keras.layers.Dense(32, activation='relu')(dense2)
output = keras.layers.Dense(1)(dense3)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=1, batch_size=64, validation_split=0.2)

# Precompute client and product embeddings
client_embeddings = model.get_layer('client_embedding').get_weights()[0]
product_embeddings = model.get_layer('product_embedding').get_weights()[0]

# Create a dataframe for predictions
predictions_df = pd.DataFrame([(client, product) for client in range(num_clients) for product in range(num_products)],
                              columns=['Client_ID', 'Product_ID'])

# Map embeddings to dataframe
predictions_df['Client_Embedding'] = predictions_df['Client_ID'].map(lambda x: client_embeddings[x])
predictions_df['Product_Embedding'] = predictions_df['Product_ID'].map(lambda x: product_embeddings[x])

# Concatenate embeddings
predictions_df['Combined_Embedding'] = predictions_df.apply(lambda row: np.concatenate([row['Client_Embedding'], row['Product_Embedding']]), axis=1)

# Convert combined embeddings to numpy array
combined_embeddings = np.vstack(predictions_df['Combined_Embedding'].values)

# Make predictions in batch
predicted_ratings = model.layers[-1](combined_embeddings)

# Assign predicted ratings to dataframe
predictions_df['Predicted_Rating'] = predicted_ratings.numpy()

# Create the client_id to original_id mapping
client_id_mapping = {encoded_id: original_id for encoded_id, original_id in zip(data['Client_ID'], client_encoder.inverse_transform(data['Client_ID']))}

# Generate top 10 recommendations for each client
recommendations = []
for client_id in range(num_clients):
    client_data = predictions_df[predictions_df['Client_ID'] == client_id]
    top_10 = client_data.sort_values(by='Predicted_Rating', ascending=False).head(10)
    
    for _, row in top_10.iterrows():
        product_name = data[data['Product_ID'] == row['Product_ID']]['Product_Name'].values[0]
        recommendations.append({
            'Client_ID': client_id_mapping[client_id],
            'Product_Name': product_name,
            'Predicted_Rating': row['Predicted_Rating'],
            'Recall_at_k': recall_at_k(y_test, row['Predicted_Rating'])
        })

recommendations_df = pd.DataFrame(recommendations)

# Save the recommendations to a CSV file
output_path = "/Users/igormol/Desktop/analytics3/sample/recommendations.csv"
recommendations_df.to_csv(output_path, index=False)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 50

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers
dense1 = keras.layers.Dense(128, activation='relu')(concat)
dense2 = keras.layers.Dense(64, activation='relu')(dense1)
dense3 = keras.layers.Dense(32, activation='relu')(dense2)
output = keras.layers.Dense(1)(dense3)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=1, batch_size=64, validation_split=0.2)

# Precompute client and product embeddings
client_embeddings = model.get_layer('client_embedding').get_weights()[0]
product_embeddings = model.get_layer('product_embedding').get_weights()[0]

# Convert combined embeddings to numpy array
def get_combined_embeddings(client_ids, product_ids):
    combined_embeddings = np.array([np.concatenate([client_embeddings[client_id], product_embeddings[product_id]])
                                    for client_id, product_id in zip(client_ids, product_ids)])
    return combined_embeddings

# Predict for the test set
combined_test_embeddings = get_combined_embeddings(X_test['Client_ID'].values, X_test['Product_ID'].values)
y_pred = model.layers[-1](combined_test_embeddings).numpy().flatten()

# Compute recall at k
def recall_at_k(y_true, y_pred, k=10):
    relevant_items = np.argsort(-y_true)[:k]
    recommended_items = np.argsort(-y_pred)[:k]
    recall = len(set(relevant_items) & set(recommended_items)) / len(relevant_items)
    return recall

# Compute recall for the test set
recalls = []
for client_id in X_test['Client_ID'].unique():
    client_data = X_test[X_test['Client_ID'] == client_id]
    y_true = y_test[client_data.index].values
    y_pred_client = model.layers[-1](get_combined_embeddings(client_data['Client_ID'].values, client_data['Product_ID'].values)).numpy().flatten()
    recalls.append(recall_at_k(y_true, y_pred_client))

average_recall = np.mean(recalls)
print(f'Average Recall at 10: {average_recall}')

# Generate mappings from encoded client IDs to original client IDs
client_id_mapping = {encoded_id: original_id for encoded_id, original_id in zip(data['Client_ID'], client_encoder.inverse_transform(data['Client_ID']))}

# Generate top 10 recommendations for all clients
client_ids = data['Client_ID'].unique()
product_ids = data['Product_ID'].unique()

recommendations = []

for client_id in client_ids:
    client_data = pd.DataFrame({'Client_ID': client_id, 'Product_ID': product_ids})
    combined_embeddings = get_combined_embeddings(client_data['Client_ID'].values, client_data['Product_ID'].values)
    predicted_ratings = model.layers[-1](combined_embeddings).numpy().flatten()
    client_data['Predicted_Rating'] = predicted_ratings
    top_10 = client_data.sort_values(by='Predicted_Rating', ascending=False).head(10)
    
    for _, row in top_10.iterrows():
        product_name = data[data['Product_ID'] == row['Product_ID']]['Product_Name'].values[0]
        recommendations.append({
            'Client_ID': client_id_mapping[int(row['Client_ID'])],
            'Product_Name': product_name,
            'Predicted_Rating': row['Predicted_Rating'],
            'Recall_at_k': recall_at_k(y_test, row['Predicted_Rating'])
        })

recommendations_df = pd.DataFrame(recommendations)

# Save the recommendations to a CSV file
output_path = "/Users/igormol/Desktop/analytics3/sample/recommendations.csv"
recommendations_df.to_csv(output_path, index=False)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 50

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers
dense1 = keras.layers.Dense(128, activation='relu')(concat)
dense2 = keras.layers.Dense(64, activation='relu')(dense1)
dense3 = keras.layers.Dense(32, activation='relu')(dense2)
output = keras.layers.Dense(1)(dense3)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=1, batch_size=64, validation_split=0.2)

# Precompute client and product embeddings
client_embeddings = model.get_layer('client_embedding').get_weights()[0]
product_embeddings = model.get_layer('product_embedding').get_weights()[0]

# Create a function to get combined embeddings
def get_combined_embeddings(client_ids, product_ids):
    client_embeds = np.array([client_embeddings[client_id] for client_id in client_ids])
    product_embeds = np.array([product_embeddings[product_id] for product_id in product_ids])
    combined_embeddings = np.concatenate([client_embeds, product_embeds], axis=1)
    return combined_embeddings

# Predict for the test set
combined_test_embeddings = get_combined_embeddings(X_test['Client_ID'].values, X_test['Product_ID'].values)
y_pred = model.predict([X_test['Client_ID'], X_test['Product_ID']]).flatten()

# Compute recall at k
def recall_at_k(y_true, y_pred, k=10):
    relevant_items = np.argsort(-y_true)[:k]
    recommended_items = np.argsort(-y_pred)[:k]
    recall = len(set(relevant_items) & set(recommended_items)) / len(relevant_items)
    return recall

# Compute recall for the test set
recalls = []
for client_id in X_test['Client_ID'].unique():
    client_data = X_test[X_test['Client_ID'] == client_id]
    y_true = y_test[client_data.index].values
    y_pred_client = model.predict([client_data['Client_ID'], client_data['Product_ID']]).flatten()
    recalls.append(recall_at_k(y_true, y_pred_client))

average_recall = np.mean(recalls)
print(f'Average Recall at 10: {average_recall}')



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 50

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers
dense1 = keras.layers.Dense(128, activation='relu')(concat)
dense2 = keras.layers.Dense(64, activation='relu')(dense1)
dense3 = keras.layers.Dense(32, activation='relu')(dense2)
output = keras.layers.Dense(1)(dense3)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=1, batch_size=64, validation_split=0.2)

# Predict for the test set using batch prediction
batch_size = 1024  # Adjust batch size according to your system's capacity
y_pred = []

for start in range(0, len(X_test), batch_size):
    end = start + batch_size
    batch_client_ids = X_test['Client_ID'][start:end]
    batch_product_ids = X_test['Product_ID'][start:end]
    batch_predictions = model.predict([batch_client_ids, batch_product_ids])
    y_pred.extend(batch_predictions.flatten())

y_pred = np.array(y_pred)

# Generate mappings from encoded client IDs to original client IDs
client_id_mapping = {encoded_id: original_id for encoded_id, original_id in zip(data['Client_ID'], client_encoder.inverse_transform(data['Client_ID']))}

# Generate top 10 recommendations for all clients
client_ids = data['Client_ID'].unique()
product_ids = data['Product_ID'].unique()

recommendations = []

# Precompute product names for faster lookups
product_id_to_name = data.set_index('Product_ID')['Product_Name'].to_dict()

# Vectorize the predictions for all clients and products
client_product_pairs = pd.DataFrame([(client_id, product_id) for client_id in client_ids for product_id in product_ids], 
                                    columns=['Client_ID', 'Product_ID'])

batch_client_ids = client_product_pairs['Client_ID']
batch_product_ids = client_product_pairs['Product_ID']

predicted_ratings = model.predict([batch_client_ids, batch_product_ids]).flatten()
client_product_pairs['Predicted_Rating'] = predicted_ratings

# Generate top 10 recommendations for each client
recommendations = []



In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 50

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers
dense1 = keras.layers.Dense(128, activation='relu')(concat)
dense2 = keras.layers.Dense(64, activation='relu')(dense1)
dense3 = keras.layers.Dense(32, activation='relu')(dense2)
output = keras.layers.Dense(1)(dense3)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=1, batch_size=64, validation_split=0.2)

# Predict for the test set without batch prediction
y_pred = model.predict([X_test['Client_ID'], X_test['Product_ID']]).flatten()

# Generate mappings from encoded client IDs to original client IDs
client_id_mapping = {encoded_id: original_id for encoded_id, original_id in zip(data['Client_ID'], client_encoder.inverse_transform(data['Client_ID']))}

# Generate all possible combinations of clients and products
client_ids = data['Client_ID'].unique()
product_ids = data['Product_ID'].unique()
client_product_combinations = np.array(np.meshgrid(client_ids, product_ids)).T.reshape(-1, 2)

# Predict ratings for all combinations at once
all_client_ids = client_product_combinations[:, 0]
all_product_ids = client_product_combinations[:, 1]
predicted_ratings = model.predict([all_client_ids, all_product_ids]).flatten()

# Create a DataFrame with all predictions
all_predictions = pd.DataFrame({
    'Client_ID': all_client_ids,
    'Product_ID': all_product_ids,
    'Predicted_Rating': predicted_ratings
})

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# For each client, get the top 10 product recommendations
recommendations = []
for client_id in client_ids:
    client_recommendations = all_predictions[all_predictions['Client_ID'] == client_id]
    top_10 = client_recommendations.nlargest(10, 'Predicted_Rating')
    for _, row in top_10.iterrows():
        recommendations.append({
            'Client_ID': client_id_mapping[int(row['Client_ID'])],
            'Product_Name': row['Product_Name'],
            'Predicted_Rating': row['Predicted_Rating']
        })

recommendations_df = pd.DataFrame(recommendations)

# Save the recommendations to a CSV file
output_path = "/Users/igormol/Desktop/analytics3/sample/recommendations.csv"
recommendations_df.to_csv(output_path, index=False)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 10

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers
dense1 = keras.layers.Dense(128, activation='relu')(concat)
dense2 = keras.layers.Dense(64, activation='relu')(dense1)
dense3 = keras.layers.Dense(32, activation='relu')(dense2)
output = keras.layers.Dense(1)(dense3)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=1, batch_size=64, validation_split=0.2)

# Generate all possible combinations of clients and products
client_ids = data['Client_ID'].unique()
product_ids = data['Product_ID'].unique()
client_product_combinations = np.array(np.meshgrid(client_ids, product_ids)).T.reshape(-1, 2)

# Predict ratings for all combinations at once
all_client_ids = client_product_combinations[:, 0]
all_product_ids = client_product_combinations[:, 1]
predicted_ratings = model.predict([all_client_ids, all_product_ids]).flatten()

# Create a DataFrame with all predictions
all_predictions = pd.DataFrame({
    'Client_ID': all_client_ids,
    'Product_ID': all_product_ids,
    'Predicted_Rating': predicted_ratings
})

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# Find top 10 recommendations for each client using groupby and apply
def get_top_10_recommendations(group):
    top_10 = group.nlargest(10, 'Predicted_Rating')
    top_10['Client_ID'] = top_10['Client_ID'].map(client_id_mapping)
    return top_10[['Client_ID', 'Product_Name', 'Predicted_Rating']]

top_recommendations = all_predictions.groupby('Client_ID').apply(get_top_10_recommendations).reset_index(drop=True)

# Save the recommendations to a CSV file
output_path = "/Users/igormol/Desktop/analytics3/sample/recommendations.csv"
top_recommendations.to_csv(output_path, index=False)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the simplified NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 20  # Reduced embedding dimension

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers
dense1 = keras.layers.Dense(64, activation='relu')(concat)  # Reduced number of units
dense2 = keras.layers.Dense(32, activation='relu')(dense1)  # Reduced number of units
output = keras.layers.Dense(1)(dense2)  # Removed one layer

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=10, batch_size=64, validation_split=0.2)

# Generate all possible combinations of clients and products
client_ids = data['Client_ID'].unique()
product_ids = data['Product_ID'].unique()
client_product_combinations = np.array(np.meshgrid(client_ids, product_ids)).T.reshape(-1, 2)

# Predict ratings for all combinations at once
all_client_ids = client_product_combinations[:, 0]
all_product_ids = client_product_combinations[:, 1]
predicted_ratings = model.predict([all_client_ids, all_product_ids]).flatten()

# Create a DataFrame with all predictions
all_predictions = pd.DataFrame({
    'Client_ID': all_client_ids,
    'Product_ID': all_product_ids,
    'Predicted_Rating': predicted_ratings
})

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# Generate mappings from encoded client IDs to original client IDs
client_id_mapping = {encoded_id: original_id for encoded_id, original_id in zip(data['Client_ID'], client_encoder.inverse_transform(data['Client_ID']))}

# Find top 10 recommendations for each client using groupby and apply
def get_top_10_recommendations(group):
    top_10 = group.nlargest(10, 'Predicted_Rating')
    top_10['Client_ID'] = top_10['Client_ID'].map(client_id_mapping)
    return top_10[['Client_ID', 'Product_Name', 'Predicted_Rating']]

top_recommendations = all_predictions.groupby('Client_ID').apply(get_top_10_recommendations).reset_index(drop=True)

# Save the recommendations to a CSV file
output_path = "/Users/igormol/Desktop/analytics3/sample/recommendations.csv"
top_recommendations.to_csv(output_path, index=False)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the simplified NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 20  # Reduced embedding dimension

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers
dense1 = keras.layers.Dense(64, activation='relu')(concat)  # Reduced number of units
dense2 = keras.layers.Dense(32, activation='relu')(dense1)  # Reduced number of units
output = keras.layers.Dense(1)(dense2)  # Removed one layer

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=1, batch_size=64, validation_split=0.2)

# Select the top N products (e.g., top 100 products)
top_n_products = 100
popular_products = data['Product_ID'].value_counts().index[:top_n_products]

# Generate combinations of clients and top N products
client_ids = data['Client_ID'].unique()
client_product_combinations = np.array([(client, product) for client in client_ids for product in popular_products])

# Predict ratings for these combinations
all_client_ids = client_product_combinations[:, 0]
all_product_ids = client_product_combinations[:, 1]
predicted_ratings = model.predict([all_client_ids, all_product_ids]).flatten()

# Create a DataFrame with all predictions and map original client IDs
client_id_mapping = dict(enumerate(client_encoder.inverse_transform(data['Client_ID'].unique())))
all_predictions = pd.DataFrame({
    'Client_ID': all_client_ids,
    'Product_ID': all_product_ids,
    'Predicted_Rating': predicted_ratings
})
all_predictions['Client_ID'] = all_predictions['Client_ID'].map(client_id_mapping)

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# Find top 10 recommendations for each client
top_recommendations = all_predictions.groupby('Client_ID').apply(
    lambda x: x.nlargest(10, 'Predicted_Rating')[['Client_ID', 'Product_Name', 'Predicted_Rating']]
).reset_index(drop=True)

# Save the recommendations to a CSV file
output_path = "/Users/igormol/Desktop/analytics3/sample/recommendations.csv"
top_recommendations.to_csv(output_path, index=False)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the simplified NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 20  # Reduced embedding dimension

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers
dense1 = keras.layers.Dense(64, activation='relu')(concat)  # Reduced number of units
dense2 = keras.layers.Dense(32, activation='relu')(dense1)  # Reduced number of units
output = keras.layers.Dense(1)(dense2)  # Removed one layer

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=1, batch_size=64, validation_split=0.2)

# Select the top N products (e.g., top 100 products)
top_n_products = 10
popular_products = data['Product_ID'].value_counts().index[:top_n_products]

# Identify the encoded Client_ID for "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_original_id = "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_encoded_id = client_encoder.transform([specific_client_original_id])[0]

# Generate combinations of the specific client and top N products
client_product_combinations = np.array([(specific_client_encoded_id, product) for product in popular_products])

# Predict ratings for these combinations
all_client_ids = client_product_combinations[:, 0]
all_product_ids = client_product_combinations[:, 1]
predicted_ratings = model.predict([all_client_ids, all_product_ids]).flatten()

# Create a DataFrame with all predictions and map original client IDs
all_predictions = pd.DataFrame({
    'Client_ID': [specific_client_original_id] * len(predicted_ratings),
    'Product_ID': all_product_ids,
    'Predicted_Rating': predicted_ratings
})

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# Find top 10 recommendations for the specific client
top_recommendations = all_predictions.nlargest(10, 'Predicted_Rating')

# Print recommendations for the specific client
print(top_recommendations)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import xgboost as xgb

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the more complex NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 50  # Increased embedding dimension

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers with increased complexity
dense1 = keras.layers.Dense(128, activation='relu')(concat)
dense2 = keras.layers.Dense(64, activation='relu')(dense1)
dense3 = keras.layers.Dense(32, activation='relu')(dense2)
output = keras.layers.Dense(1)(dense3)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=5, batch_size=64, validation_split=0.2)

# Get the embeddings and predictions for the training set
embeddings_train = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_train['Client_ID'], X_train['Product_ID']])
predictions_train = model.predict([X_train['Client_ID'], X_train['Product_ID']])

# Train the XGBoost model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=6)
xgb_model.fit(embeddings_train, y_train)

# Predict ratings for the test set
embeddings_test = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_test['Client_ID'], X_test['Product_ID']])
xgb_predictions = xgb_model.predict(embeddings_test)

# Evaluate the performance
mse = np.mean((xgb_predictions - y_test) ** 2)
print(f'Mean Squared Error: {mse}')

# Identify the encoded Client_ID for "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_original_id = "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_encoded_id = client_encoder.transform([specific_client_original_id])[0]

# Generate combinations of the specific client and all products
client_product_combinations = np.array([(specific_client_encoded_id, product) for product in range(num_products)])

# Predict ratings for these combinations using the neural network
all_client_ids = client_product_combinations[:, 0]
all_product_ids = client_product_combinations[:, 1]
embeddings_comb = keras.models.Model(inputs=model.inputs, outputs=concat).predict([all_client_ids, all_product_ids])
nn_predictions = model.predict([all_client_ids, all_product_ids]).flatten()

# Predict ratings using the XGBoost model
xgb_predictions_comb = xgb_model.predict(embeddings_comb)

# Create a DataFrame with all predictions and map original client IDs
all_predictions = pd.DataFrame({
    'Client_ID': [specific_client_original_id] * len(xgb_predictions_comb),
    'Product_ID': all_product_ids,
    'Predicted_Rating': xgb_predictions_comb
})

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# Find top 10 recommendations for the specific client
top_recommendations = all_predictions.nlargest(10, 'Predicted_Rating')

# Print recommendations for the specific client
print(top_recommendations)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import xgboost as xgb

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the significantly more complex NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 100  # Increased embedding dimension

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers with significantly increased complexity and regularization
dense1 = keras.layers.Dense(1024, activation='relu')(concat)
dropout1 = keras.layers.Dropout(0.5)(dense1)
dense2 = keras.layers.Dense(512, activation='relu')(dropout1)
dropout2 = keras.layers.Dropout(0.5)(dense2)
dense3 = keras.layers.Dense(256, activation='relu')(dropout2)
dropout3 = keras.layers.Dropout(0.5)(dense3)
dense4 = keras.layers.Dense(128, activation='relu')(dropout3)
dropout4 = keras.layers.Dropout(0.5)(dense4)
dense5 = keras.layers.Dense(64, activation='relu')(dropout4)
dropout5 = keras.layers.Dropout(0.5)(dense5)
output = keras.layers.Dense(1)(dropout5)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=10, batch_size=64, validation_split=0.2)

# Get the embeddings and predictions for the training set
embeddings_train = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_train['Client_ID'], X_train['Product_ID']])
predictions_train = model.predict([X_train['Client_ID'], X_train['Product_ID']])

# Train the significantly more complex XGBoost model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=1000, learning_rate=0.01, max_depth=10, subsample=0.8, colsample_bytree=0.8)
xgb_model.fit(embeddings_train, y_train)

# Predict ratings for the test set
embeddings_test = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_test['Client_ID'], X_test['Product_ID']])
xgb_predictions = xgb_model.predict(embeddings_test)

# Evaluate the performance
mse = np.mean((xgb_predictions - y_test) ** 2)
print(f'Mean Squared Error: {mse}')

# Identify the encoded Client_ID for "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_original_id = "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_encoded_id = client_encoder.transform([specific_client_original_id])[0]

# Generate combinations of the specific client and all products
client_product_combinations = np.array([(specific_client_encoded_id, product) for product in range(num_products)])

# Predict ratings for these combinations using the neural network
all_client_ids = client_product_combinations[:, 0]
all_product_ids = client_product_combinations[:, 1]
embeddings_comb = keras.models.Model(inputs=model.inputs, outputs=concat).predict([all_client_ids, all_product_ids])
nn_predictions = model.predict([all_client_ids, all_product_ids]).flatten()

# Predict ratings using the XGBoost model
xgb_predictions_comb = xgb_model.predict(embeddings_comb)

# Create a DataFrame with all predictions and map original client IDs
all_predictions = pd.DataFrame({
    'Client_ID': [specific_client_original_id] * len(xgb_predictions_comb),
    'Product_ID': all_product_ids,
    'Predicted_Rating': xgb_predictions_comb
})

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# Find top 10 recommendations for the specific client
top_recommendations = all_predictions.nlargest(10, 'Predicted_Rating')

# Print recommendations for the specific client
print(top_recommendations)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import xgboost as xgb

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the less complex NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 10  # Reduced embedding dimension

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers with reduced complexity and regularization
dense1 = keras.layers.Dense(64, activation='relu')(concat)
dropout1 = keras.layers.Dropout(0.2)(dense1)
dense2 = keras.layers.Dense(32, activation='relu')(dropout1)
dropout2 = keras.layers.Dropout(0.2)(dense2)
output = keras.layers.Dense(1)(dropout2)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=5, batch_size=64, validation_split=0.2)

# Get the embeddings and predictions for the training set
embeddings_train = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_train['Client_ID'], X_train['Product_ID']])
predictions_train = model.predict([X_train['Client_ID'], X_train['Product_ID']])

# Train the XGBoost model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=1000, learning_rate=0.01, max_depth=6, subsample=0.8, colsample_bytree=0.8)
xgb_model.fit(embeddings_train, y_train)

# Predict ratings for the test set
embeddings_test = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_test['Client_ID'], X_test['Product_ID']])
xgb_predictions = xgb_model.predict(embeddings_test)

# Evaluate the performance
mse = np.mean((xgb_predictions - y_test) ** 2)
print(f'Mean Squared Error: {mse}')

# Identify the encoded Client_ID for "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_original_id = "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_encoded_id = client_encoder.transform([specific_client_original_id])[0]

# Generate combinations of the specific client and all products
client_product_combinations = np.array([(specific_client_encoded_id, product) for product in range(num_products)])

# Predict ratings for these combinations using the neural network
all_client_ids = client_product_combinations[:, 0]
all_product_ids = client_product_combinations[:, 1]
embeddings_comb = keras.models.Model(inputs=model.inputs, outputs=concat).predict([all_client_ids, all_product_ids])
nn_predictions = model.predict([all_client_ids, all_product_ids]).flatten()

# Predict ratings using the XGBoost model
xgb_predictions_comb = xgb_model.predict(embeddings_comb)

# Create a DataFrame with all predictions and map original client IDs
all_predictions = pd.DataFrame({
    'Client_ID': [specific_client_original_id] * len(xgb_predictions_comb),
    'Product_ID': all_product_ids,
    'Predicted_Rating': xgb_predictions_comb
})

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# Find top 10 recommendations for the specific client
top_recommendations = all_predictions.nlargest(10, 'Predicted_Rating')

# Print recommendations for the specific client
print(top_recommendations)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import xgboost as xgb

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the slightly simpler NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 9  # Reduced embedding dimension by 10%

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers with slightly reduced complexity and regularization
dense1 = keras.layers.Dense(115, activation='relu')(concat)  # Reduced units by ~10%
dropout1 = keras.layers.Dropout(0.2)(dense1)
dense2 = keras.layers.Dense(58, activation='relu')(dropout1)  # Reduced units by ~10%
dropout2 = keras.layers.Dropout(0.2)(dense2)
output = keras.layers.Dense(1)(dropout2)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=5, batch_size=64, validation_split=0.2)

# Get the embeddings and predictions for the training set
embeddings_train = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_train['Client_ID'], X_train['Product_ID']])
predictions_train = model.predict([X_train['Client_ID'], X_train['Product_ID']])

# Train the XGBoost model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=90, learning_rate=0.1, max_depth=6)  # Slightly reduced n_estimators
xgb_model.fit(embeddings_train, y_train)

# Predict ratings for the test set
embeddings_test = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_test['Client_ID'], X_test['Product_ID']])
xgb_predictions = xgb_model.predict(embeddings_test)

# Evaluate the performance
mse = np.mean((xgb_predictions - y_test) ** 2)
print(f'Mean Squared Error: {mse}')

# Identify the encoded Client_ID for "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_original_id = "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_encoded_id = client_encoder.transform([specific_client_original_id])[0]

# Generate combinations of the specific client and all products
client_product_combinations = np.array([(specific_client_encoded_id, product) for product in range(num_products)])

# Predict ratings for these combinations using the neural network
all_client_ids = client_product_combinations[:, 0]
all_product_ids = client_product_combinations[:, 1]
embeddings_comb = keras.models.Model(inputs=model.inputs, outputs=concat).predict([all_client_ids, all_product_ids])
nn_predictions = model.predict([all_client_ids, all_product_ids]).flatten()

# Predict ratings using the XGBoost model
xgb_predictions_comb = xgb_model.predict(embeddings_comb)

# Normalize the predicted ratings to a range between 0 and 100
scaler = MinMaxScaler(feature_range=(0, 100))
xgb_predictions_comb_normalized = scaler.fit_transform(xgb_predictions_comb.reshape(-1, 1)).flatten()

# Create a DataFrame with all predictions and map original client IDs
all_predictions = pd.DataFrame({
    'Client_ID': [specific_client_original_id] * len(xgb_predictions_comb_normalized),
    'Product_ID': all_product_ids,
    'Predicted_Rating': xgb_predictions_comb_normalized
})

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# Find top 10 recommendations for the specific client
top_recommendations = all_predictions.nlargest(10, 'Predicted_Rating')

# Print recommendations for the specific client
print(top_recommendations)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import xgboost as xgb
from concurrent.futures import ThreadPoolExecutor, as_completed

# Load the data
data_path = "/Users/igormol/Desktop/analytics3/sample/Marajo.csv"
data = pd.read_csv(data_path)

# Fill missing values (e.g., brand names) if necessary
data['Brand_Name'].fillna('Unknown', inplace=True)

# Encode Client_ID and Product_ID
client_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data['Client_ID'] = client_encoder.fit_transform(data['Client_ID'])
data['Product_ID'] = product_encoder.fit_transform(data['Product_ID'])

# Extract features and target
X = data[['Client_ID', 'Product_ID']]
y = data['Monetary_Value']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the slightly simpler NCF model
num_clients = data['Client_ID'].nunique()
num_products = data['Product_ID'].nunique()

# Embedding dimension
embedding_dim = 9  # Reduced embedding dimension by 10%

# Client embedding
client_input = keras.layers.Input(shape=(1,), name='client_input')
client_embedding = keras.layers.Embedding(input_dim=num_clients, output_dim=embedding_dim, name='client_embedding')(client_input)
client_vec = keras.layers.Flatten()(client_embedding)

# Product embedding
product_input = keras.layers.Input(shape=(1,), name='product_input')
product_embedding = keras.layers.Embedding(input_dim=num_products, output_dim=embedding_dim, name='product_embedding')(product_input)
product_vec = keras.layers.Flatten()(product_embedding)

# Concatenate embeddings
concat = keras.layers.Concatenate()([client_vec, product_vec])

# Fully connected layers with slightly reduced complexity and regularization
dense1 = keras.layers.Dense(115, activation='relu')(concat)  # Reduced units by ~10%
dropout1 = keras.layers.Dropout(0.2)(dense1)
dense2 = keras.layers.Dense(58, activation='relu')(dropout1)  # Reduced units by ~10%
dropout2 = keras.layers.Dropout(0.2)(dense2)
output = keras.layers.Dense(1)(dropout2)

# Define the model
model = keras.models.Model(inputs=[client_input, product_input], outputs=output)
model.compile(optimizer='adam', loss='mse')

# Train the model
history = model.fit([X_train['Client_ID'], X_train['Product_ID']], y_train, epochs=5, batch_size=64, validation_split=0.2)

# Get the embeddings and predictions for the training set
embeddings_train = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_train['Client_ID'], X_train['Product_ID']])
predictions_train = model.predict([X_train['Client_ID'], X_train['Product_ID']])

# Train the XGBoost model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=90, learning_rate=0.1, max_depth=6)  # Slightly reduced n_estimators
xgb_model.fit(embeddings_train, y_train)

# Predict ratings for the test set
embeddings_test = keras.models.Model(inputs=model.inputs, outputs=concat).predict([X_test['Client_ID'], X_test['Product_ID']])
xgb_predictions = xgb_model.predict(embeddings_test)

# Evaluate the performance
mse = np.mean((xgb_predictions - y_test) ** 2)
print(f'Mean Squared Error: {mse}')

# Identify the encoded Client_ID for "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_original_id = "40.146.358 ADRIANA CAMPOS MATOS"
specific_client_encoded_id = client_encoder.transform([specific_client_original_id])[0]

# Generate combinations of the specific client and all products
client_product_combinations = np.array([(specific_client_encoded_id, product) for product in range(num_products)])

def predict_for_combination(client_product_pair):
    client_id, product_id = client_product_pair
    embedding = keras.models.Model(inputs=model.inputs, outputs=concat).predict([np.array([client_id]), np.array([product_id])])
    nn_prediction = model.predict([np.array([client_id]), np.array([product_id])]).flatten()[0]
    xgb_prediction = xgb_model.predict(embedding.reshape(1, -1))[0]  # Reshape the embedding for XGBoost prediction
    return product_id, xgb_prediction

# Use ThreadPoolExecutor to parallelize the predictions
with ThreadPoolExecutor(max_workers=8) as executor:
    future_to_product = {executor.submit(predict_for_combination, pair): pair for pair in client_product_combinations}
    predictions = []
    for future in as_completed(future_to_product):
        product_id, prediction = future.result()
        predictions.append((specific_client_encoded_id, product_id, prediction))

# Normalize the predicted ratings to a range between 0 and 100
predictions_df = pd.DataFrame(predictions, columns=['Client_ID', 'Product_ID', 'Predicted_Rating'])
scaler = MinMaxScaler(feature_range=(0, 100))
predictions_df['Predicted_Rating'] = scaler.fit_transform(predictions_df[['Predicted_Rating']])

# Create a DataFrame with all predictions and map original client IDs
all_predictions = predictions_df
all_predictions['Client_ID'] = specific_client_original_id

# Merge to get product names
all_predictions = all_predictions.merge(data[['Product_ID', 'Product_Name']].drop_duplicates(), on='Product_ID', how='left')

# Find top 10 recommendations for the specific client
top_recommendations = all_predictions.nlargest(10, 'Predicted_Rating')

# Print recommendations for the specific client
print(top_recommendations)
